In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}

def add_panel_labels(figure):
    panels = [axis for axis in figure.axes if axis.get_label() != "<colorbar>"]
    assert len(panels) == 15, "Unexpected panel count"
    for index, axis in enumerate(panels):
        axis.text(-0.16, 1.04, f"({chr(97 + index)})",
                  transform=axis.transAxes, fontsize=12, fontweight="bold", fontfamily="DejaVu Serif",
                  ha="left", va="bottom", clip_on=False, gid="panel_label")
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    records = []
    for axis in panels:
        label = next(text for text in axis.texts if text.get_gid() == "panel_label")
        box = label.get_window_extent(renderer)
        for title in (axis.title, axis._left_title, axis._right_title):
            if title.get_text():
                assert not box.overlaps(title.get_window_extent(renderer)), "Panel label overlaps title"
        records.append({"label": label.get_text(), "axes_position": list(axis.get_position().bounds),
                        "label_bbox_pixels": list(box.bounds)})

def export_figure(figure, figure_id, tight=False):
    add_panel_labels(figure)
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
metadata=json.loads((PACKAGE_ROOT/"data/FigS1_FigS2/metadata.json").read_text())
means={}
for key in metadata:
    values=load_npz(f"data/FigS1_FigS2/{key}.npz")["amplitude"]
    assert len(values)==metadata[key]["included_runs"]
    means[key]=values.mean(axis=0)
xf=np.arange(40)
freq_y_mean_noiseless=means["noiseless"]
freq_y_mean_noise=means["noise"]
freq_y_mean_noise2=means["noise2"]
freq_y_mean_noise3=means["noise3"]
freq_y_mean_noise4=means["noise4"]
freq_y_mean_noise5=means["noise5"]
print("Actual saved runs:",{key:rec["included_runs"] for key,rec in metadata.items()})

In [ ]:
from matplotlib import font_manager
font_manager.fontManager.addfont(str(PACKAGE_ROOT/"data/FigS1_FigS2/fonts/NimbusRoman-Regular.ttf"))
import matplotlib.cm as cm
plt.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"font.family":"Nimbus Roman","font.size":14,"axes.linewidth":1.2})
def plot_fig_epoch_triple_appendix(freq_index_list, filename_index = None):
    fig, axs = plt.subplots(3, 5, figsize=(14, 9), sharey=True)
    wndex = 0
    for jndex in range(0,3):
        for index in range(5):
            freq_index = freq_index_list[wndex]
            wndex += 1
            ax = axs[jndex,index]
            
            ax.plot([freq_y_mean_noiseless[wndex][freq_index] for wndex in range(600)], label="Noiseless", color='orange',linewidth=0.7)
            ax.plot([freq_y_mean_noise[wndex][freq_index] for wndex in range(600)], label="$p_x = p_y = 0, p_z = 0.15$", color='blue',linewidth=0.7)
            ax.plot([freq_y_mean_noise2[wndex][freq_index] for wndex in range(600)], label="$p_x = p_y = 0, p_z = 0.30$", color='green',linewidth=0.7)
            ax.plot([freq_y_mean_noise3[wndex][freq_index] for wndex in range(600)], label="$p_x = p_y = p_z = 0.05$", color='red',linewidth=0.7)
            if jndex == 2:
                ax.set_xlabel("Training Iteration")
            
            ax.set_title(rf"$k={int(xf[freq_index])}$")
            if index == 0 and jndex == 0:

                ax.legend()
                ax.legend(loc='upper center', bbox_to_anchor=(2.9, 1.35), ncol=4, fontsize=14, frameon=False)
                
            if index == 0:
                ax.set_ylabel("Amplitude")
        
    plt.subplots_adjust(hspace=0.3)
    plt.tight_layout()
    return fig

In [ ]:
fig=plot_fig_epoch_triple_appendix([1,2,4,5,6,7,9,10,11,12,13,14,15,20,21])
export_figure(fig,"FigS1",tight=True)
plt.close(fig)